# Food orders analysis (pandas)

In [27]:
import numpy as np
import pandas as pd

## PART A: Load and inspect

In [28]:
print("=== PART A ===")
df = pd.read_csv("food_orders.csv")

print("A1 First 5 rows:")
print(df.head(5))
print("A1 Last 3 rows:")
print(df.tail(3))

print("\nA2 Shape:", df.shape)
print("A2 Columns:", list(df.columns))
print("A2 Dtypes:")
print(df.dtypes)

print("\nA3 info():")
df.info()
print("A3 describe():")
print(df.describe())

print("\nA4 Missing values per column:")
print(df.isnull().sum())
print("A4 Repeated Order_IDs:", df["Order_ID"].duplicated().sum())

=== PART A ===
A1 First 5 rows:
   Order_ID   Customer_Name         City          Restaurant       Cuisine  \
0  ORD10124     Neha Sharma      Chennai           Dum House       Biryani   
1  ORD10296     divya hegde  hyderabad    Filter Kaapi House  South Indian   
2  ORD10230    Pooja Iyer      Hyderabad      Biryani Bazaar       Biryani   
3  ORD10158     anjali nair   Bangalore        Dosa Junction  South Indian   
4  ORD10172    Divya Shetty    Bengaluru         Cake Studio      Desserts   

   Order_Date  Order_Value  Delivery_Time_Min  Rating Payment_Mode  \
0  2026-10-18        688.0                NaN     4.0         Card   
1  2026-10-02        267.0               35.0     4.0         Card   
2  2026-10-12        507.0               36.0     NaN         Card   
3  2026-10-09        125.0               55.0     4.0         Cash   
4  2026-10-12        230.0               42.0     3.0  Credit Card   

  Internal_Ref  
0    REF518129  
1    REF343052  
2    REF339360  
3    REF78

## PART B: Clean the data (in order)

In [29]:
print("\n=== PART B ===")

df = df.drop_duplicates(subset="Order_ID", keep="first")
print("B1 Shape after removing duplicates:", df.shape)

df["Customer_Name"] = df["Customer_Name"].str.strip().str.title()
df["City"] = df["City"].str.strip().str.title()
print("B2 Cities after strip/title:", sorted(df["City"].unique()))

city_map = {
    "Bangalore": "Bengaluru",
    "Mysore": "Mysuru",
    "Mangalore": "Mangaluru",
    "Hubli": "Hubballi",
}
df["City"] = df["City"].replace(city_map)
print("B3 Cities after standardising:", sorted(df["City"].unique()))

payment_map = {
    "upi": "UPI",
    "card": "Card",
    "debit card": "Card",
    "credit card": "Card",
    "cod": "Cash on Delivery",
    "cash": "Cash on Delivery",
    "cash on delivery": "Cash on Delivery",
}
print("B4 Payment spellings before:", df["Payment_Mode"].nunique())

df["Payment_Mode"] = df["Payment_Mode"].str.strip().str.lower()
df["Payment_Mode"] = df["Payment_Mode"].replace(payment_map)
print("B4 Payment values after:", df["Payment_Mode"].value_counts().to_dict())

delivery_mean = df["Delivery_Time_Min"].mean()
rating_mean = df["Rating"].mean()
df["Delivery_Time_Min"] = df["Delivery_Time_Min"].fillna(delivery_mean)
df["Rating"] = df["Rating"].fillna(rating_mean)
print("B5 Delivery_Time_Min mean used:", delivery_mean)
print("B5 Rating mean used:", rating_mean)

df = df.dropna()
print("B6 Shape after dropna:", df.shape)

df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%Y-%m-%d")
df["Month"] = df["Order_Date"].dt.month
df["Day_Name"] = df["Order_Date"].dt.day_name()
print("B7 Order_Date dtype:", df["Order_Date"].dtype)
print(df[["Order_Date", "Month", "Day_Name"]].head(3))

df = df.rename(columns={"Order_Value": "Order_Value_INR"})
df = df.drop(columns=["Internal_Ref"])
df = df.reset_index(drop=True)
print("B8 Final columns:", list(df.columns))
print("B8 Final shape:", df.shape)
print("B8 Missing values left:", df.isnull().sum().sum())


=== PART B ===
B1 Shape after removing duplicates: (310, 11)
B2 Cities after strip/title: ['Bangalore', 'Bengaluru', 'Chennai', 'Hubballi', 'Hubli', 'Hyderabad', 'Mangalore', 'Mangaluru', 'Mysore', 'Mysuru']
B3 Cities after standardising: ['Bengaluru', 'Chennai', 'Hubballi', 'Hyderabad', 'Mangaluru', 'Mysuru']
B4 Payment spellings before: 12
B4 Payment values after: {'UPI': 148, 'Card': 84, 'Cash on Delivery': 78}
B5 Delivery_Time_Min mean used: 38.1280276816609
B5 Rating mean used: 4.043321299638989
B6 Shape after dropna: (302, 11)
B7 Order_Date dtype: datetime64[us]
  Order_Date  Month Day_Name
0 2026-10-18     10   Sunday
1 2026-10-02     10   Friday
2 2026-10-12     10   Monday
B8 Final columns: ['Order_ID', 'Customer_Name', 'City', 'Restaurant', 'Cuisine', 'Order_Date', 'Order_Value_INR', 'Delivery_Time_Min', 'Rating', 'Payment_Mode', 'Month', 'Day_Name']
B8 Final shape: (302, 12)
B8 Missing values left: 0


## PART C: Analysis

In [30]:
print("\n=== PART C ===")

c1 = df[(df["City"] == "Bengaluru") & (df["Order_Value_INR"] > 800) & (df["Rating"] >= 4)]
print("C1 count:", len(c1))

c2 = df[(df["Cuisine"].isin(["Biryani", "Chinese"])) & (df["Payment_Mode"] != "UPI")]
print("C2 shape:", c2.shape)

c3 = df.nlargest(5, "Order_Value_INR")[["Order_ID", "City", "Cuisine", "Order_Value_INR"]]
print("C3 Top 5 orders by value:")
print(c3)

c4 = df.sort_values(["City", "Order_Value_INR"], ascending=[True, False])
print("C4 Sorted by City A-Z, value high-low (first 8):")
print(c4.head(8))

df["Delivery_Status"] = np.where(df["Delivery_Time_Min"] <= 40, "On time", "Late")
print("C5 Delivery_Status counts:", df["Delivery_Status"].value_counts().to_dict())

def rating_band(rating):
    if rating >= 4.5:
        return "Excellent"
    elif rating >= 4:
        return "Good"
    elif rating >= 3:
        return "Average"
    return "Poor"


df["Rating_Band"] = df["Rating"].apply(rating_band)
print("C6 Rating_Band counts:", df["Rating_Band"].value_counts().to_dict())

c7 = df.groupby("City").agg(
    Avg_Delivery_Min=("Delivery_Time_Min", "mean"),
    Orders=("Order_ID", "count"),
)
print("C7 Average delivery time and order count per city:")
print(c7)

c8 = df.groupby("Cuisine")["Order_Value_INR"].agg(["mean", "max", "min", "count"])
print("C8 Order value stats per cuisine:")
print(c8)


=== PART C ===
C1 count: 11
C2 shape: (66, 12)
C3 Top 5 orders by value:
     Order_ID       City  Cuisine  Order_Value_INR
132  ORD10174  Hyderabad  Biryani           1086.0
41   ORD10308  Hyderabad  Biryani           1077.0
117  ORD10147  Hyderabad  Biryani           1072.0
289  ORD10237    Chennai  Biryani           1062.0
160  ORD10226  Mangaluru  Biryani           1041.0
C4 Sorted by City A-Z, value high-low (first 8):
     Order_ID    Customer_Name       City      Restaurant       Cuisine  \
286  ORD10141    Karthik Reddy  Bengaluru    Nawabi Handi       Biryani   
73   ORD10032       Divya Nair  Bengaluru       Dum House       Biryani   
268  ORD10134      Anjali Naik  Bengaluru       Dum House       Biryani   
299  ORD10092    Vivaan Shetty  Bengaluru    Nawabi Handi       Biryani   
56   ORD10135      Kavya Reddy  Bengaluru  Biryani Bazaar       Biryani   
7    ORD10227  Ishaan Kulkarni  Bengaluru       Dum House       Biryani   
155  ORD10031      Sneha Menon  Bengaluru    N

## PART D: Export and verify

In [31]:
print("\n=== PART D ===")
df.to_csv("cleaned_orders.csv", index=False)
df.to_json("cleaned_orders.json", orient="records", indent=4, date_format="iso")
print("D1 Saved cleaned_orders.csv and cleaned_orders.json")

check = pd.read_json("cleaned_orders.json")
print("D2 JSON shape:", check.shape, "| Cleaned DataFrame shape:", df.shape)
print("D2 Match:", check.shape == df.shape)


=== PART D ===
D1 Saved cleaned_orders.csv and cleaned_orders.json
D2 JSON shape: (302, 14) | Cleaned DataFrame shape: (302, 14)
D2 Match: True


## PART E: Insights

In [32]:
# 1) Bengaluru is the slowest city: average delivery is 44.49 min (C7),
#    while Mysuru is the fastest at 30.54 min.
# 2) Biryani has the highest average order value: 717.15 INR (C8),
#    South Indian is the lowest at 281.32 INR.
# 3) 116 of 302 orders were Late (C5), which is about 38.4% of all orders.